# Conformational Differences and Embedding Distances

## Setup

In [ ]:
from setup import *

import matplotlib.colors as mcolors
from torch.utils.data import Subset, random_split

## Compute Conformer Correlations

In [ ]:
def correlation_table(records, labels):
    table = pd.DataFrame({"Comparison": labels, "Molecules": records["n_common"].to_numpy()})
    for model in ("ANT", "NT", "nopre"):
        table[f"{model}: median rho (95% CI)"] = [
            f"{row[model]:.3f} ({row[model + '_low']:.3f}, {row[model + '_high']:.3f})"
            if np.isfinite(row[model]) else "Undefined"
            for _, row in records.iterrows()
        ]
    return table

# Additional geometric measures and molecule-level summaries.

METRICS = (
    ("rmsd_all", "All-atom RMSD", "RMSD (angstrom)"),
    ("rmsd_heavy", "Heavy-atom RMSD", "RMSD (angstrom)"),
    ("rmsd_symmetry", "Symmetry-aware RMSD", "RMSD (angstrom)"),
    ("torsion", "Torsional distance", "RMS angular difference (rad)"),
)

def _selected_smiles(directory, selected_ids):
    lookup = {}
    wanted = set(selected_ids)
    for path in sorted(Path(directory).glob("*_id_smiles_fingerprint.npy")):
        data = np.load(path, mmap_mode="r", allow_pickle=False)
        for row in data[np.isin(data["id"], list(wanted))]:
            decode = lambda value: value.decode("utf-8") if isinstance(value, bytes) else str(value)
            lookup[int(row["id"])] = {
                "smiles": decode(row["smiles"]),
                "chembl_id": decode(row["chembl_id"]) if "chembl_id" in data.dtype.names else "",
            }
    missing = wanted - lookup.keys()
    if missing:
        raise ValueError(f"Missing SMILES for {len(missing)} selected molecules in {directory}.")
    return lookup

def geometric_comparisons(dataset, molecule_ids, smiles_directory, max_symmetry_matches=10000):
    """Additional distances for the same molecules and conformer pairs as the heatmap."""
    from rdkit import Chem
    from rdkit.Chem import Lipinski, rdMolTransforms
    from tqdm.auto import tqdm

    lookup = _selected_smiles(smiles_directory, molecule_ids)
    pairs = np.triu_indices(len(dataset[0]), k=1)
    arrays = {key: [] for key in ("rmsd_heavy", "rmsd_symmetry", "torsion")}
    metadata = []
    for row, molecule_id in tqdm(enumerate(molecule_ids), total=len(dataset),
                                 desc="Symmetry and torsion comparisons"):
        conformers = dataset[row]
        coords = np.stack([conf.coord.cpu().numpy() for conf in conformers])
        numbers = conformers[0].numbers.cpu().numpy()
        info = lookup[molecule_id]
        mol = Chem.AddHs(Chem.MolFromSmiles(info["smiles"]))
        if not np.array_equal(numbers, [atom.GetAtomicNum() for atom in mol.GetAtoms()]):
            raise ValueError(f"SMILES/coordinate atom ordering differs for molecule {molecule_id}.")
        heavy = Chem.RemoveAllHs(mol)
        heavy_coords = coords[:, numbers > 1, :]
        if heavy.GetNumAtoms() != heavy_coords.shape[1]:
            raise ValueError(f"Heavy-atom correspondence differs for molecule {molecule_id}.")
        fixed = conformer_rmsd(heavy_coords, pairs)
        arrays["rmsd_heavy"].append(fixed)
        matches = heavy.GetSubstructMatches(
            heavy, uniquify=False, useChirality=True, maxMatches=max_symmetry_matches + 1
        ) if heavy.GetNumAtoms() else ()
        symmetry = np.full(len(pairs[0]), np.nan)
        symmetry_status = "no_heavy_atoms"
        if len(matches) > max_symmetry_matches:
            symmetry_status = "match_limit_exceeded"
        elif matches:
            # Minimize over chirality-preserving graph automorphisms and proper rotations.
            symmetry = fixed.copy()
            for match in matches:
                symmetry = np.minimum(symmetry, conformer_rmsd(heavy_coords, pairs, match))
            symmetry_status = "ok"
        arrays["rmsd_symmetry"].append(symmetry)

        bonds = sorted({tuple(sorted(match)) for match in heavy.GetSubstructMatches(
            Lipinski.RotatableBondSmarts
        )})
        dihedrals = []
        for b, c in bonds:
            left = [a.GetIdx() for a in heavy.GetAtomWithIdx(b).GetNeighbors() if a.GetIdx() != c]
            right = [a.GetIdx() for a in heavy.GetAtomWithIdx(c).GetNeighbors() if a.GetIdx() != b]
            if left and right:
                dihedrals.append((min(left), b, c, min(right)))
        angles = np.empty((len(conformers), len(dihedrals)))
        for conf_index, positions in enumerate(heavy_coords):
            conf = Chem.Conformer(len(positions))
            for atom_index, position in enumerate(positions):
                conf.SetAtomPosition(atom_index, tuple(map(float, position)))
            for angle_index, atoms in enumerate(dihedrals):
                angles[conf_index, angle_index] = rdMolTransforms.GetDihedralRad(conf, *atoms)
        arrays["torsion"].append(conformer_torsion_distances(angles, pairs))
        metadata.append({
            **info, "molecule_id": molecule_id, "dataset_index": dataset.indices[row],
            "heavy_atoms": int(np.count_nonzero(numbers > 1)),
            "rotatable_bonds": len(bonds), "torsions": len(dihedrals),
            "symmetry_matches": len(matches), "symmetry_status": symmetry_status,
            "torsion_status": "ok" if len(dihedrals) and np.isfinite(angles).all()
                              else "no_torsions_or_undefined_angles",
        })
    arrays = {key: np.stack(value) for key, value in arrays.items()}
    return arrays, metadata

def paired_bootstrap(a, b, n_bootstrap=2000, seed=0, nopre=None):
    """Resample shared valid molecules jointly; summarize model medians and ANT-NT changes."""
    valid = np.isfinite(a) & np.isfinite(b)
    if nopre is not None:
        valid &= np.isfinite(nopre)
    a, b = np.asarray(a)[valid], np.asarray(b)[valid]
    names = ["ANT", "NT", "difference"]
    values = [a, b, a - b]
    if nopre is not None:
        names.append("nopre")
        values.append(np.asarray(nopre)[valid])
    result = {"n_common": len(a)}
    if not len(a):
        return {**result, **{name + suffix: np.nan for name in names
                            for suffix in ("", "_low", "_high")}}
    medians = [np.median(value) for value in values]
    bootstrap = []
    rng = np.random.default_rng(seed)
    if len(a) > 1:
        for start in range(0, n_bootstrap, 128):
            indices = rng.integers(0, len(a), size=(min(128, n_bootstrap - start), len(a)))
            bootstrap.append(np.column_stack([np.median(value[indices], axis=1)
                                              for value in values]))
        intervals = np.percentile(np.concatenate(bootstrap), (2.5, 97.5), axis=0)
    else:
        intervals = np.full((2, len(names)), np.nan)
    for index, name in enumerate(names):
        result[name] = float(medians[index])
        result[name + "_low"] = float(intervals[0, index])
        result[name + "_high"] = float(intervals[1, index])
    return result

def summarize_correlations(geometries, embeddings, molecules, n_bootstrap=2000, seed=0):
    correlations = {
        (metric, model): spearman_correlations(geometries[metric], values)
        for metric, _, _ in METRICS for model, values in embeddings.items()
    }
    per_molecule = []
    for index, info in enumerate(molecules):
        record = dict(info)
        for metric, _, _ in METRICS:
            for model in embeddings:
                record[f"rho_{metric}_{model}"] = correlations[(metric, model)][index]
        per_molecule.append(record)

    heavy = np.asarray([m["heavy_atoms"] for m in molecules])
    rotors = np.asarray([m["rotatable_bonds"] for m in molecules])
    groups = [ ("All", "All", np.ones(len(molecules), dtype=bool)),
               ("Heavy atoms", "<=20", heavy <= 20),
               ("Heavy atoms", "21-40", (heavy > 20) & (heavy <= 40)),
               ("Heavy atoms", ">40", heavy > 40),
               ("Rotatable bonds", "0", rotors == 0),
               ("Rotatable bonds", "1-3", (rotors >= 1) & (rotors <= 3)),
               ("Rotatable bonds", "4-7", (rotors >= 4) & (rotors <= 7)),
               ("Rotatable bonds", ">=8", rotors >= 8) ]
    summaries = []
    for metric, _, _ in METRICS:
        for feature, label, mask in groups:
            a = correlations[(metric, "ANT")][mask]
            b = correlations[(metric, "NT")][mask]
            c = correlations[(metric, "nopre")][mask]
            summaries.append({"metric": metric, "feature": feature, "group": label,
                              "n_molecules": int(mask.sum()),
                              "n_valid_ANT": int(np.isfinite(a).sum()),
                              "n_valid_NT": int(np.isfinite(b).sum()),
                              "n_valid_nopre": int(np.isfinite(c).sum()),
                              **paired_bootstrap(a, b, n_bootstrap, seed, nopre=c)})

    return per_molecule, summaries

sample_per_batch = 16
dataset = ConformerEdgeDataset(str(CHEMBL_DIR), num_conformers=16, sample_per_batch=sample_per_batch, use_molecule_properties=False, randomize=False, preload=False)
# Same molecular split as train.py; all conformers of a molecule stay together.
# The validation molecules were excluded from weight updates and used for checkpoint selection.
# Reproducing this split requires the same dataset contents and ordering as pretraining.
_, val_dataset = random_split(dataset, [0.95, 0.05], generator=torch.Generator().manual_seed(0))
sample_indices = np.sort(np.random.default_rng(0).choice(
    val_dataset.indices, size=min(1000, len(val_dataset)), replace=False))
dataset = Subset(dataset, sample_indices.tolist())
del val_dataset, _
model = EncoderModel.load_model(CHECKPOINTS["ANT"], device=DEVICE).eval()
rmsd_ANT, emb_dist_ANT, ids_ANT = compute_conformer_distances(
    model, dataset, ConformerEdge_collate_fn, DEVICE, sample_per_batch, batch_size=16, num_workers=4)
model = EncoderModel.load_model(CHECKPOINTS["NT"], device=DEVICE).eval()
rmsd_NT, emb_dist_NT, ids_NT = compute_conformer_distances(
    model, dataset, ConformerEdge_collate_fn, DEVICE, sample_per_batch, batch_size=16, num_workers=4)
model = EncoderModel.load_model(CHECKPOINTS["nopre"], device=DEVICE).eval()
rmsd_nopre, emb_dist_nopre, ids_nopre = compute_conformer_distances(
    model, dataset, ConformerEdge_collate_fn, DEVICE, sample_per_batch, batch_size=16, num_workers=4)

assert torch.equal(ids_ANT, ids_NT) and torch.equal(ids_ANT, ids_nopre), "All three models must use the same molecules."
assert len(torch.unique(ids_ANT)) == len(dataset), "Molecule IDs must be unique."
assert torch.allclose(rmsd_ANT, rmsd_NT) and torch.allclose(rmsd_ANT, rmsd_nopre), "All three models must use the same conformer pairs."
assert all(torch.isfinite(x).all() and (x >= 0).all() and (x <= 2).all()
           for x in (emb_dist_ANT, emb_dist_NT, emb_dist_nopre)), "Invalid cosine distances."

rmsd_flat_ANT, rmsd_flat_NT, rmsd_flat_nopre = rmsd_ANT.flatten().numpy(), rmsd_NT.flatten().numpy(), rmsd_nopre.flatten().numpy()
emb_dist_flat_ANT, emb_dist_flat_NT, emb_dist_flat_nopre = emb_dist_ANT.flatten().numpy(), emb_dist_NT.flatten().numpy(), emb_dist_nopre.flatten().numpy()

# Tables use the same molecules and all 120 conformer pairs for all three models.
extra_geometry, molecule_metadata = geometric_comparisons(dataset, ids_ANT.tolist(), str(SMILES_DIR))
conformer_correlations, conformer_summary = summarize_correlations(
    {"rmsd_all": rmsd_ANT.numpy(), **extra_geometry},
    {"ANT": emb_dist_ANT.numpy(), "NT": emb_dist_NT.numpy(), "nopre": emb_dist_nopre.numpy()},
    molecule_metadata, n_bootstrap=2000, seed=0,
)

## Figure 6: Conformer RMSD and Embedding Distance

In [ ]:
def plot_heatmap(ax, rmsd_flat, emb_dist_flat, title, vmin, vmax, bins):

    h = ax.hist2d(rmsd_flat, emb_dist_flat, bins=bins,
                   cmap="viridis", norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))
    ax.set_title(title, fontsize=6, pad=0)
    ax.set_xlim(0, bins[0][-1])
    ax.set_ylim(0, bins[1][-1])
    ax.set_xticks(np.linspace(0, bins[0][-1], 5))
    ax.set_yticks(np.linspace(0, bins[1][-1], 6))
    ax.tick_params(labelsize=4, length=2, width=0.4, pad=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.4)
    return h

# shared color scale across both panels so they're visually comparable
fig, axes = plt.subplots(3, 1, figsize=(7.1 * cm, 7.1 * cm), sharex=True)
bbox = axes[0].get_window_extent().transformed(fig.dpi_scale_trans.inverted())
width, height = int(bbox.width * 600) // 8, int(bbox.height * 600) // 8 + 10 # looks square enough skull
rmsd_max = max(8.0, np.ceil(max(rmsd_flat_ANT.max(), rmsd_flat_NT.max(), rmsd_flat_nopre.max())))
embedding_max = max(1.0, emb_dist_flat_ANT.max(), emb_dist_flat_NT.max(), emb_dist_flat_nopre.max())
bins = [np.linspace(0, rmsd_max, width + 1), np.linspace(0, embedding_max, height + 1)]
all_counts_ANT, _, _ = np.histogram2d(rmsd_flat_ANT, emb_dist_flat_ANT, bins=bins)
all_counts_NT, _, _ = np.histogram2d(rmsd_flat_NT, emb_dist_flat_NT, bins=bins)
all_counts_nopre, _, _ = np.histogram2d(rmsd_flat_nopre, emb_dist_flat_nopre, bins=bins)
vmin = 1
vmax = max(2, all_counts_ANT.max(), all_counts_NT.max(), all_counts_nopre.max())
plot_heatmap(axes[0], rmsd_flat_ANT, emb_dist_flat_ANT, MODEL_TITLES["ANT"], vmin, vmax, bins)
axes[0].tick_params(axis="x", which="both", bottom=True, labelbottom=False)
plot_heatmap(axes[1], rmsd_flat_NT, emb_dist_flat_NT, MODEL_TITLES["NT"], vmin, vmax, bins)
axes[1].tick_params(axis="x", which="both", bottom=True, labelbottom=False)
heatmap = plot_heatmap(axes[2], rmsd_flat_nopre, emb_dist_flat_nopre, MODEL_TITLES["nopre"], vmin, vmax, bins)
axes[2].set_xlabel("Kabsch RMSD (Angstroms)", fontsize=6, ha="center", va="center")
fig.text(0.02, 0.5, "Embedding Distance (1 - Cosine)", fontsize=6, rotation=90, ha="center", va="center")
fig.subplots_adjust(hspace=0.15, left=0.09, top=0.95, bottom=0.08, right=0.85)

# single shared colorbar for both panels
cbar_ax = fig.add_axes([0.87, 0.15, 0.03, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(heatmap[3], cax=cbar_ax)
cbar.ax.set_xlabel("count", fontsize=6, labelpad=4, ha="center")
cbar.ax.xaxis.set_label_coords(0.5, cbar.ax.xaxis.get_label().get_position()[1])
cbar.ax.tick_params(labelsize=4, length=2, width=0.4)
cbar.outline.set_linewidth(0.4)
fig.savefig(OUTPUT_DIR / "rmsd_embedding_heatmap.png", dpi=600)
plt.show()

## Table 1: Within-Molecule Geometric Correlations

In [ ]:
records = pd.DataFrame(conformer_summary)
geometry_records = records[records["feature"] == "All"]
display(correlation_table(geometry_records, [title for _, title, _ in METRICS]))

## Table S7: Correlations by Molecular Size and Flexibility

In [ ]:
records = pd.DataFrame(conformer_summary)
strata_records = records[(records["metric"] == "rmsd_all") & (records["feature"] != "All")]
display(correlation_table(strata_records, [f"{row['feature']}: {row['group']}" for _, row in strata_records.iterrows()]))